# Introduction to Python, Part IV: NumPy

In Parts I through III we covered Python's built-in tools: variables, operators, control flow, functions, and the four core containers (lists, tuples, sets, and dictionaries). This notebook introduces **NumPy**, the standard library for numerical computing in Python. NumPy gives us the `array`, a container built specifically for fast, vectorized math on whole collections of numbers at once, which makes it the natural tool for the vectors, matrices, and functions you work with in a math class.

Each section starts with a short explanation, often including the **general form** of the syntax, followed by a code cell you can run and experiment with. Try changing the values in the code cells to see how the output changes, that's the best way to learn!

# Installing and Importing NumPy

Unlike `print()` or lists, NumPy is not part of core Python: it's a third-party package that needs to be installed once before you can use it. From a terminal, you'd run:

```
pip install numpy
```

Inside a Jupyter notebook, you can run the same kind of command directly in a code cell using `%pip install numpy`. The `%` prefix is an IPython "magic" command; it installs into the exact same environment your notebook's kernel is using. This is safer than `!pip install numpy` (running pip as a plain shell command), which can silently install into a different Python than the one running your notebook if your system has more than one. You only need to run this once per environment; if NumPy is already installed (as it usually is in a course environment), the command below will simply confirm that and exit quickly.

After installing, you import NumPy like any other module. By convention, it's almost always imported under the short alias `np`.

In [ ]:
%pip install numpy # Run once per environment; safe to re-run, it will just confirm numpy is installed

import numpy as np # The standard convention: import numpy under the alias np

print(np.__version__) # Print the installed version of numpy

# Sequences with `np.arange`

You already know Python's built-in `range(start, stop, step)` from Parts I and II. NumPy has its own version, `np.arange`, with the same general form:

```
np.arange(start, stop, step)
```

`start` defaults to `0`, `step` defaults to `1`, and `stop` is always exclusive, exactly like `range()`. There are two key differences, though:

- `range()` produces a special, memory-light `range` object containing only integers; you need `list(...)` to see its values. `np.arange()` immediately returns an actual NumPy array, ready to use in math.
- `np.arange()` accepts non-integer steps, e.g. `np.arange(0, 1, 0.1)`, which `range()` cannot do at all.

Because of this, `np.arange` is usually the more convenient way to build a numeric array from scratch when you're already working with NumPy.

In [ ]:
r = range(0, 10, 2) # A built-in range object
print(r, type(r)) # Print the value of r and its type
print(list(r)) # Convert to a list to see the actual values

a = np.arange(0, 10, 2) # The NumPy equivalent, returned directly as an array
print(a, type(a)) # Print the value of a and its type

b = np.arange(5) # start defaults to 0, step defaults to 1
print(b) # Print the array

c = np.arange(0, 1, 0.1) # A float step: not possible with range()
print(c) # Print the array of evenly spaced float values

# Sequences with `np.linspace`

`np.linspace` is a close relative of `np.arange`, but instead of specifying the step size, you specify how many points you want. Its general form:

```
np.linspace(start, stop, num)
```

`num` (default `50`) is the number of evenly spaced points to generate between `start` and `stop`. The key difference from `np.arange` is that `linspace` includes `stop` by default, it's inclusive, whereas `arange`'s `stop` is exclusive. This makes `linspace` the natural choice whenever you're thinking "I want exactly `num` points across this interval," which comes up constantly when sampling a function to evaluate or plot it. You can exclude the endpoint with `endpoint=False` if you need `arange`-style behavior.

In [ ]:
a = np.arange(0, 1, 0.25) # Step of 0.25: stop (1) is NOT included
print(a) # Print the array

b = np.linspace(0, 1, 5) # 5 evenly spaced points: stop (1) IS included
print(b) # Print the array

c = np.linspace(0, 1, 5, endpoint=False) # Same 5 points, but excluding the endpoint
print(c) # Print the array

x = np.linspace(0, 2 * np.pi, 9) # 9 evenly spaced angles covering a full period
print(x) # Print the angles

y = np.sin(x) # Evaluate sin(x) at each of those 9 points
print(y) # Print the resulting sine values

# Creating Arrays

The core object in NumPy is the array (`np.ndarray`), an ordered grid of values, all of the same type (usually numbers). Arrays can be one-dimensional (like a vector), two-dimensional (like a matrix), or higher-dimensional.

The simplest way to create one is with `np.array`, which converts a Python list (or a list of lists, for a matrix) into an array. Its general form is:

```
np.array(object, dtype=None)
```

`object` is the list (or nested list) of values, and `dtype` optionally forces the array to use a specific data type (e.g. `int`, `float`), rather than letting NumPy guess one for you.

In [ ]:
v = np.array([1, 2, 3, 4, 5]) # Create a 1D array (a vector) from a list
print(v, type(v)) # Print the value of v and its type

print(v.shape) # shape is a tuple describing the size along each dimension
print(v.dtype) # dtype is the data type of the elements: integers by default here

M = np.array([[1, 2, 3], [4, 5, 6]]) # Create a 2D array (a matrix) from a list of lists
print(M) # Print the matrix
print(M.shape) # (2, 3): 2 rows, 3 columns

By default, NumPy infers the `dtype` from your data: a list of whole numbers becomes integers, but if any value in the list is a float (or you pass `dtype=float` explicitly), the whole array becomes floats instead.

This matters more than it might seem. An integer array can only ever hold whole numbers: assigning a decimal value into it silently truncates the decimal part instead of rounding or raising an error. If you plan to store or compute decimal results later (averages, square roots, solutions to equations), it's safer to create the array with `dtype=float` from the start, even if your initial values happen to be whole numbers.

In [ ]:
int_arr = np.array([1, 2, 3]) # No dtype given: NumPy infers integers from whole numbers
print(int_arr, int_arr.dtype) # Print the array and its inferred dtype

int_arr[0] = 3.9 # Try to store a decimal value in an integer array
print(int_arr) # The 3.9 was silently truncated to 3, not rounded!

float_arr = np.array([1, 2, 3], dtype=float) # Explicitly force a float dtype
print(float_arr, float_arr.dtype) # Print the array and its dtype: float64

float_arr[0] = 3.9 # Now the same assignment works correctly
print(float_arr) # 3.9 is stored exactly, as intended

Two other common ways to create arrays are `np.zeros` and `np.ones`, which build an array of a given shape filled entirely with `0`s or `1`s. Their general forms are:

```
np.zeros(shape, dtype=float)
np.ones(shape, dtype=float)
```

`shape` can be a single number (for a 1D array) or a tuple like `(rows, cols)` (for a 2D array); note that, unlike `np.array`, these default to `dtype=float` already. These are useful as starting points, for example, initializing a vector of coefficients before filling in real values.

In [ ]:
z = np.zeros(5) # A 1D array of 5 zeros
print(z) # Print the array

Z = np.zeros((2, 4)) # A 2x4 matrix of zeros
print(Z) # Print the matrix

o = np.ones(5) # A 1D array of 5 ones
print(o) # Print the array

O = np.ones((3, 3)) # A 3x3 matrix of ones
print(O) # Print the matrix

Sometimes you want an array filled with a specific value other than `0` or `1`, or you want to overwrite the contents of an array you already have. `np.full` creates a new array filled with a chosen value; `.fill()` overwrites every entry of an existing array in place. Their general forms:

```
np.full(shape, fill_value, dtype=None)
arr.fill(value)
```

In [ ]:
sevens = np.full(5, 7) # A 1D array of five 7s
print(sevens) # Print the array

nines = np.full((3, 3), 9) # A 3x3 matrix filled with 9
print(nines) # Print the matrix

existing = np.zeros((2, 3)) # Start with a 2x3 matrix of zeros
print(existing) # Print the original matrix

existing.fill(4) # Overwrite every entry in place with 4
print(existing) # Print the updated matrix

# Arithmetic with Arrays

One of the main reasons to use NumPy arrays instead of plain lists is that arithmetic operators work elementwise: `+`, `-`, `*`, `/`, and `**` are applied to each pair of matching entries at once, no loop required. Compare this to a regular Python list, where `+` means concatenation and `*` by a scalar means repetition, not math!

Arithmetic between an array and a single number (a scalar) applies the operation to every element (this is called broadcasting).

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

print(a + b) # Elementwise addition: [1+10, 2+20, 3+30]
print(b - a) # Elementwise subtraction
print(a * b) # Elementwise multiplication (NOT a matrix product, see below for that)
print(b / a) # Elementwise division
print(a ** 2) # Elementwise squaring

print(a * 10) # Broadcasting: multiply every element by the scalar 10
print(a + 1) # Broadcasting: add 1 to every element

list_a = [1, 2, 3]
list_b = [10, 20, 30]
print(list_a + list_b) # For plain lists, '+' means concatenation, not addition!

# NumPy Math Functions

NumPy provides vectorized versions of common math functions, `np.sin`, `np.cos`, `np.sqrt`, `np.exp`, `np.log`, and many more, which apply the function to every element of an array at once. This is especially useful for evaluating a mathematical function at many points, for example, sampling $\sin(x)$ at several angles.

In [ ]:
angles = np.arange(0, np.pi / 2 + 0.01, np.pi / 6) # A few evenly spaced angles in radians
print(angles) # Print the array of angles

sines = np.sin(angles) # Evaluate sin(x) at every angle at once
print(sines) # Print the resulting sine values

cosines = np.cos(angles) # Evaluate cos(x) at every angle at once
print(cosines) # Print the resulting cosine values

values = np.array([1, 4, 9, 16, 25])
roots = np.sqrt(values) # Elementwise square root
print(roots) # Print the square roots

exps = np.exp(np.array([0, 1, 2])) # Elementwise e^x
print(exps) # Print the exponentials

logs = np.log(np.array([1, np.e, np.e ** 2])) # Elementwise natural log
print(logs) # Print the logarithms

# Writing Your Own Functions with Array Inputs

Since arrays behave like regular values, you can write your own functions that take an array as a parameter and use NumPy operations inside the function body, exactly as you defined functions with numbers or lists in Part II. This lets you package a reusable calculation, like standardizing a set of data, behind a single name.

Recall that the z-score (or standardized value) of a dataset rescales it to have mean 0 and standard deviation 1, by subtracting the mean and dividing by the standard deviation: $z = \dfrac{x - \bar{x}}{s}$.

In [ ]:
def standardize(x): # x is expected to be a NumPy array
    return (x - x.mean()) / x.std() # NumPy's elementwise math does the work for every entry at once

data = np.array([2, 4, 4, 4, 5, 5, 7, 9])
z = standardize(data) # Call the function on our array
print(z) # Print the standardized values

print(z.mean()) # Should be (very close to) 0
print(z.std()) # Should be (very close to) 1

def vector_length(v): # v is expected to be a 1D NumPy array (a vector)
    return np.sqrt(np.sum(v ** 2)) # sqrt of the sum of squared components

print(vector_length(np.array([3, 4]))) # Should be 5.0: a 3-4-5 triangle

# Random Number Generators

NumPy's `random` module generates random numbers, useful for simulations, sampling, and statistics. The recommended modern approach is to create a generator with `np.random.default_rng(seed)`.

Passing a fixed `seed` makes the sequence of "random" numbers reproducible: running the cell again with the same seed gives you the same values, which is very useful when you want a repeatable experiment or homework answer.

In [ ]:
rng = np.random.default_rng(seed=42) # Create a random generator with a fixed seed

uniform_sample = rng.random(5) # 5 random floats, uniformly distributed in [0, 1)
print(uniform_sample) # Print the sample

dice_rolls = rng.integers(low=1, high=7, size=10) # 10 random integers from 1 to 6 (like rolling a die)
print(dice_rolls) # Print the rolls

normal_sample = rng.normal(loc=0, scale=1, size=5) # 5 samples from a standard normal distribution
print(normal_sample) # Print the sample

normal_sample_mean_10 = rng.normal(loc=10, scale=2, size=5) # mean 10, standard deviation 2
print(normal_sample_mean_10) # Print the sample

# Indexing and Slicing

1D arrays support the same indexing and slicing syntax as lists:

```
arr[i]                # a single element
arr[start:stop:step]  # a slice
```

with negative indices and negative steps for reversing, exactly as in Part I.

In [ ]:
v = np.array([10, 20, 30, 40, 50])
print(v[0]) # First element
print(v[-1]) # Last element
print(v[1:4]) # Elements at positions 1, 2, 3
print(v[::2]) # Every other element
print(v[::-1]) # The array reversed

2D arrays (matrices) add a second index for the column, with the general form:

```
M[row, col]                                # a single element
M[row_start:row_stop: row_step, col_start:col_stop:col_step]  # a sub-block
```

You can also select a whole row with `M[row, :]` (or just `M[row]`), and a whole column with `M[:, col]`.

In [ ]:
M = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) # A 3x3 matrix
print(M) # Print the matrix

print(M[0, 2]) # Row 0, column 2
print(M[1, :]) # The whole second row (index 1)
print(M[:, 0]) # The whole first column (index 0)
print(M[0:2, 0:2]) # The top-left 2x2 sub-block

Arrays with three or more dimensions are often called tensors. A 3D tensor can be thought of as a stack of matrices; indexing extends naturally with one index per dimension:

```
T[i, j, k]                                         # a single element
T[i_start:i_stop, j_start:j_stop, k_start:k_stop]  # a sub-block
```

The first index selects which matrix in the stack, and the remaining two behave just like 2D indexing within that matrix.

In [ ]:
T = np.arange(24).reshape(2, 3, 4) # A 3D tensor: 2 matrices, each 3x4
print(T) # Print the whole tensor
print(T.shape) # (2, 3, 4)

print(T[0]) # The first 3x4 matrix in the stack
print(T[0, 1]) # Row 1 of the first matrix
print(T[0, 1, 2]) # A single element: matrix 0, row 1, column 2
print(T[:, :, 0]) # The first column of every matrix in the stack
print(T[1, 0:2, 1:3]) # A sub-block: matrix 1, rows 0-1, columns 1-2

Indexing retrieves the value at a position you already know. Often you want the opposite: given a condition, find *where* it holds. `np.where(condition)` returns the indices where `condition` is `True`, its general form:

```
np.where(condition)
```

`condition` doesn't have to be an equality check, it can be any comparison, such as `x > n`. For a 1D array, `np.where` returns a tuple containing one array of matching indices; for a 2D array, it returns a tuple of two arrays, one of row indices and one of matching column indices, paired up position by position. `np.argwhere(condition)` does the same search but returns the results already paired up as an array of `[row, col]` index pairs, which is often more convenient to read or loop over. This is the NumPy analogue of a list's `.index()` method, except it finds every match at once, not just the first.

In [ ]:
v = np.array([10, 20, 30, 20, 40]) # A 1D array with a repeated value

result = np.where(v == 20) # Find every index where the value is 20
print(result) # Print the raw result: a tuple containing an array of indices

indices = result[0] # Extract the array of matching indices
print(indices) # Print just the indices

first_index = indices[0] # The first matching index, like list.index() would give
print(first_index) # Print it

above_20 = np.where(v > 20) # The condition doesn't have to be equality
print(above_20[0]) # Print the indices where v is greater than 20

M = np.array([[1, 2, 3], [4, 5, 6], [3, 8, 3]]) # A 2D array with a repeated value
print(M) # Print the matrix

rows, cols = np.where(M == 3) # Row indices and column indices of every 3, paired up
print(rows) # Print the row indices
print(cols) # Print the matching column indices

for r, c in zip(rows, cols): # Loop over the matching (row, column) pairs together
    print(f"Found a 3 at row {r}, column {c}")

pairs = np.argwhere(M > 4) # Same idea, but already paired up as [row, col] entries
print(pairs) # Print the array of index pairs where M is greater than 4

There's an even more direct way to pull out the values that satisfy a condition, without needing their positions at all: **boolean masking**. Writing a condition like `x > n` on an array doesn't just give one `True`/`False`, it gives a whole new array of `True`/`False` values, one per entry, called a **boolean mask**. Indexing the original array with that mask, `x[x > n]`, keeps only the entries where the mask is `True`. General form:

```
x[condition]
```

where `condition` is any comparison built from the array itself, such as `x > n`, `x == n`, or a combination with `&` (and) / `|` (or). This works for 2D arrays too, though the result is always returned as a flat 1D array, since a filtered-out matrix generally isn't rectangular anymore.

In [ ]:
x = np.array([3, -1, 4, -1, 5, -9, 2, 6])

mask = x > 0 # A boolean array: True wherever the condition holds
print(mask) # Print the mask itself

positive_values = x[mask] # Keep only the entries where mask is True
print(positive_values) # Print the filtered array

positive_values2 = x[x > 0] # The same thing, written in one line without naming the mask
print(positive_values2) # Print the filtered array

evens = x[x % 2 == 0] # Any condition works, not just comparisons to a number
print(evens) # Print only the even values

middle = x[(x > -5) & (x < 5)] # Combine conditions with & (and) / | (or); each side needs its own parentheses
print(middle) # Print values strictly between -5 and 5

x[x < 0] = 0 # You can also assign through a mask: replace every negative value with 0
print(x) # Print the updated array

M = np.array([[1, -2, 3], [-4, 5, -6]]) # A 2D array
print(M[M > 0]) # Boolean masking on a 2D array always returns a flat 1D result

# Reshaping

`arr.reshape(new_shape)` rearranges the same data into a new shape, as long as the total number of elements stays the same (a 12-element array can become a `3x4` matrix, a `4x3` matrix, or a `2x2x3` tensor, but not a `5x2`). The general form is:

```
arr.reshape(new_shape)
np.reshape(arr, new_shape)
```

You can also use `-1` for one dimension to let NumPy figure it out automatically.

In [ ]:
flat = np.arange(12) # A 1D array with values 0 through 11
print(flat) # Print the flat array

grid = flat.reshape(3, 4) # Reshape into a 3x4 matrix
print(grid) # Print the reshaped matrix

grid2 = flat.reshape(4, 3) # Same data, reshaped into a 4x3 matrix instead
print(grid2) # Print the reshaped matrix

grid3 = flat.reshape(2, -1) # -1 means "figure out this dimension automatically"
print(grid3) # Print the reshaped matrix: 2 rows, 6 columns inferred

back_to_flat = grid.reshape(-1) # Flatten a matrix back into a 1D array
print(back_to_flat) # Print the flattened array

# Transpose

The transpose of a matrix flips it over its diagonal, turning rows into columns and columns into rows. The general form:

```
arr.T
np.transpose(arr)
```

In [ ]:
M = np.array([[1, 2, 3], [4, 5, 6]]) # A 2x3 matrix
print(M) # Print the original matrix
print(M.shape) # (2, 3)

M_T = M.T # Transpose using the .T attribute
print(M_T) # Print the transposed matrix
print(M_T.shape) # (3, 2): rows and columns are swapped

M_T2 = np.transpose(M) # Equivalent, using the np.transpose() function
print(M_T2) # Print the transposed matrix

# Matrix Multiplication

Recall that `*` between two arrays is elementwise multiplication, not the matrix product from linear algebra. For true matrix multiplication (where the number of columns of the first matrix must match the number of rows of the second), use:

```
A @ B
np.matmul(A, B)
np.dot(A, B)
```

all three are equivalent for two matrices. For example, multiplying a $2\times 2$ matrix by a column vector applies a linear transformation to that vector.

In [ ]:
A = np.array([[1, 2], [3, 4]]) # A 2x2 matrix
B = np.array([[5, 6], [7, 8]]) # Another 2x2 matrix

elementwise = A * B # Elementwise product, NOT the matrix product
print(elementwise) # Print the elementwise product

matrix_product = A @ B # True matrix multiplication using the @ operator
print(matrix_product) # Print the matrix product

matrix_product2 = np.matmul(A, B) # Equivalent, using np.matmul()
print(matrix_product2) # Print the matrix product

x = np.array([1, 0]) # A column vector, written as a 1D array
transformed = A @ x # Apply the matrix A to the vector x
print(transformed) # Print the transformed vector

# Sum, Max, Min, and the `axis` Argument

`np.sum`, `np.max`, and `np.min` (also available as methods, `arr.sum()`, `arr.max()`, `arr.min()`) combine an array's values into a single summary number by default. Their general form:

```
arr.sum(axis=None)
arr.max(axis=None)
arr.min(axis=None)
```

For a 2D array, the `axis` argument lets you aggregate along just one dimension instead of over the whole array:
- `axis=0` collapses down the rows, giving one result per column.
- `axis=1` collapses across the columns, giving one result per row.

In [ ]:
scores = np.array([[78, 92, 65], [88, 100, 71], [95, 60, 83]]) # 3 students (rows) x 3 quizzes (cols)
print(scores) # Print the score matrix

total = scores.sum() # Sum of every entry in the matrix
print(total) # Print the overall total

highest = scores.max() # Largest single score in the whole matrix
lowest = scores.min() # Smallest single score in the whole matrix
print(highest, lowest) # Print the max and min

quiz_totals = scores.sum(axis=0) # Sum DOWN each column: total per quiz
print(quiz_totals) # Print one total per quiz

student_totals = scores.sum(axis=1) # Sum ACROSS each row: total per student
print(student_totals) # Print one total per student

best_per_quiz = scores.max(axis=0) # Highest score achieved on each quiz
print(best_per_quiz) # Print the best score per quiz

best_per_student = scores.max(axis=1) # Each student's best quiz score
print(best_per_student) # Print the best score per student

# Reverse and Flip

You already saw that slicing with a step of `-1` (`arr[::-1]`) reverses a 1D array, just like it does for lists. For arrays with more than one dimension, `np.flip` is a clearer, explicit way to reverse the order of elements:

```
np.flip(arr, axis=None)
```

Without an `axis` argument, it reverses along every dimension at once. With `axis=0`, it flips the order of the rows (upside down); with `axis=1`, it flips the order of the columns (left to right).

In [ ]:
v = np.array([1, 2, 3, 4, 5])
print(v[::-1]) # Reverse a 1D array using slicing
print(np.flip(v)) # Equivalent, using np.flip()

M = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(M) # Print the original matrix

print(np.flip(M)) # Flip along every dimension: reverses rows AND columns
print(np.flip(M, axis=0)) # Flip along axis 0: reverses the ROW order (upside down)
print(np.flip(M, axis=1)) # Flip along axis 1: reverses the COLUMN order (left to right)

# Unique Values

`np.unique` returns the distinct values in an array, sorted in ascending order, with duplicates removed. Its general form:

```
np.unique(arr, return_counts=False)
```

It's the NumPy analogue of converting a list to a set, but the result comes back sorted, and as an array rather than a set. Passing `return_counts=True` also gives you how many times each unique value appeared.

In [ ]:
remainders = np.array([2, 0, 1, 2, 2, 0, 3, 1, 0]) # Remainders computed in a loop, with repeats

distinct = np.unique(remainders) # Sorted, distinct values only
print(distinct) # Print the distinct values

values, counts = np.unique(remainders, return_counts=True) # Also get how many times each occurred
print(values) # Print the distinct values
print(counts) # Print how many times each value appeared, in the same order as values

## Summary

In this notebook you learned how to:
- Install NumPy with `%pip install numpy` and import it as `np`
- Generate sequences with `np.arange(start, stop, step)`, and how it compares to Python's built-in `range()`
- Generate a fixed number of evenly spaced points with `np.linspace(start, stop, num)`, and how it compares to `np.arange`
- Create arrays with `np.array()` (from a list or list of lists), `np.zeros()`, and `np.ones()`
- Fill an array with a chosen value using `np.full()`, or overwrite an existing array's contents with `.fill()`
- Set and reason about an array's `dtype`, and why an explicit `dtype=float` matters when you need decimal precision
- Perform elementwise arithmetic (`+ - * / **`) and understand broadcasting with scalars
- Apply vectorized math functions like `np.sin()`, `np.cos()`, `np.sqrt()`, `np.exp()`, and `np.log()`
- Write your own functions that take a NumPy array as input and perform math operations on it
- Generate random numbers with `np.random.default_rng()`, including reproducible results using a seed
- Index and slice 1D, 2D, and 3D (tensor) arrays, including selecting rows, columns, and sub-blocks
- Find the location of values satisfying a condition with `np.where()` and `np.argwhere()`, for both 1D and 2D arrays
- Filter an array down to the values satisfying a condition with boolean masking, `x[x > n]`
- Reshape arrays with `.reshape()`, including using `-1` to infer a dimension
- Transpose a matrix with `.T` or `np.transpose()`
- Perform true matrix multiplication with `@`, `np.matmul()`, or `np.dot()`, as opposed to elementwise `*`
- Aggregate arrays with `.sum()`, `.max()`, and `.min()`, using `axis` to aggregate along rows or columns
- Reverse and flip arrays with slicing and `np.flip()`
- Find distinct values with `np.unique()`, including counts

Together with Parts I through III, you now have the core Python and NumPy tools needed to work with numerical and mathematical data.

## Practice Questions

The five questions below cover everything in this notebook (arrays, `arange`, dtype, arithmetic, functions, random numbers, slicing, tensors, reshaping, transpose, matrix multiplication, aggregation, flipping, and unique values), ordered from easy to hard.

- New to NumPy? Focus on Q1 and Q2 (and Q3 if you have time).
- Already comfortable with arrays? Start from Q4 and work through Q5 as well.

For each question, write your code in the cell provided and make sure it prints your final answer.

### Q1: Arrays, arange, and arithmetic

Use `np.arange` to create an array `x` containing the integers from 1 to 10. Compute and print `x + 5`, `x * 2`, and `x ** 2`. Then use `np.sqrt()` to print the square root of every element, and check its `dtype`; think about why it isn't an integer array.

In [ ]:
# TODO: your code for Q1 here

### Q2: Evaluating a function and slicing

Create an array `x` of 9 evenly spaced values from `0` to `2 * np.pi` using `np.linspace(0, 2 * np.pi, 9)`. Compute `y = np.sin(x)` and print both arrays. Then, using slicing, print only the first three values of `y`, only the last three, and every other value.

In [ ]:
# TODO: your code for Q2 here

### Q3: Random sampling with a function and a loop

Using `np.random.default_rng(seed=0)`, generate an array of 200 samples from a normal distribution with mean `5` and standard deviation `2`. Write a function `count_above(arr, threshold)` that uses a `for` loop over the elements of `arr` (e.g. `for value in arr:`) to count how many exceed `threshold`. Call your function on the samples with a threshold of your choice, then check your answer against the faster, vectorized approach `np.sum(arr > threshold)`, they should match.

In [ ]:
# TODO: your code for Q3 here

### Q4: Reshaping and a row-by-row function

Create a 1D array with the integers from 1 to 20 using `np.arange(1, 21)`, then reshape it into a `4x5` matrix `M`. Write a function `row_summary(M)` that loops over the rows of `M` with a `for` loop (`for row in M:`) and, for each row, prints its sum and its max using an f-string. Call your function on `M`. Then verify your printed values are correct by computing `M.sum(axis=1)` and `M.max(axis=1)` directly and comparing.

In [ ]:
# TODO: your code for Q4 here

### Q5: Matrix multiplication with a function and a loop

Write a function `apply_matrix(A, vectors)` that takes a matrix `A` and a list of 1D arrays `vectors`, and returns a new list containing `A @ v` for every `v` in `vectors`, computed using a `for` loop over the list (not a single combined matrix operation). Create a `2x2` matrix `A` of your choice and a list of three different 2D vectors, then call your function and print each transformed vector. Finally, print `A` flipped along `axis=0` and along `axis=1`.

In [ ]:
A = np.array([[0, -1], [1, 0]]) # A 90-degree rotation matrix
vectors = [np.array([1, 0]), np.array([0, 1]), np.array([2, 3])]

# TODO: your code for Q5 here